In [1]:
# NativeNext management setup; original course cells follow with the explicitly documented batch-parameter override.
import sys
sys.path.insert(0,'C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261021T021303-7a4dad65')
import next_identity as _identity
_identity.write_child_runtime_identity()
import next_adapter as _next
_next.install('C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261021T021303-7a4dad65\\source\\HaiLong_Colab_Exercises','C:\\Users\\[LEARNER_NAME_OMITTED]\\.v\\hi\\20261021T021303-7a4dad65','04_fastai_part2/14_augment.ipynb')


# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Full training can exceed a free Colab session; QUICK_RUN does not shorten these original Part 2 experiments.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/14_augment.ipynb

In [2]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '14_augment.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: C:\Users\[LOCAL_USER]\.v\hi\20261021T021303-7a4dad65\source\HaiLong_Colab_Exercises\04_fastai_part2
Run mode: FULL EXERCISE
Results: C:\Users\[LOCAL_USER]\.v\hi\20261021T021303-7a4dad65\source\HaiLong_Colab_Exercises\04_fastai_part2\results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [3]:
MAX_BATCH_SIZE = 1024  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

In [4]:
#|default_exp augment

# Augmentation

In [5]:
#|export
import torch,random
import fastcore.all as fc

from torch import nn
from torch.nn import init

from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *

In [6]:
import pickle,gzip,math,os,time,shutil
import matplotlib as mpl,numpy as np,matplotlib.pyplot as plt
from collections.abc import Mapping
from pathlib import Path
from operator import attrgetter,itemgetter
from functools import partial
from copy import copy
from contextlib import contextmanager

import torchvision.transforms.functional as TF,torch.nn.functional as F
from torch import tensor,optim
from torch.utils.data import DataLoader,default_collate
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from datasets import load_dataset,load_dataset_builder

from fastcore.test import test_close
from torch import distributions

torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'

import logging
logging.disable(logging.WARNING)

set_seed(42)

if fc.defaults.cpus>2: fc.defaults.cpus=2

In [7]:
xl,yl = 'image','label'
name = "zalando-datasets/fashion_mnist"
bs = min(1024, MAX_BATCH_SIZE)
xmean,xstd = 0.28, 0.35

@inplace
def transformi(b): b[xl] = [(TF.to_tensor(o)-xmean)/xstd for o in b[xl]]

dsd = load_dataset(name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

README.md: 0.00B [00:00, ?B/s]

C:\Users\[LOCAL_USER]\.v\hlp213\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\[LOCAL_USER]\.v\hi\20261021T021303-7a4dad65\private_cache\hf\hub\datasets--zalando-datasets--fashion_mnist. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train-00000-of-00001.parquet:   0%|          | 0.00/30.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [8]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
astats = ActivationStats(fc.risinstance(GeneralRelu))
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), astats]
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)

In [9]:
set_seed(42)
lr,epochs = 6e-2,5

## Going wider

In [10]:
def get_model(act=nn.ReLU, nfs=(16,32,64,128,256,512), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [nn.Flatten(), nn.Linear(nfs[-1], 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [11]:
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [12]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 519.079 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.000 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 519.079 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.824,0.704,0,train
0.857,0.574,0,eval
0.897,0.382,1,train
0.891,0.344,1,eval
0.923,0.265,2,train
0.907,0.296,2,eval
0.941,0.197,3,train
0.928,0.239,3,eval
0.962,0.140,4,train
0.933,0.225,4,eval


Storage check (checkpoint_write): artifacts 0.020 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.926 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.059 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.887 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.118 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.772 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.118 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.772 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.118 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.776 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.118 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.776 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.118 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.776 GB. Budget is not Drive quota.


### Pooling

In [13]:
class GlobalAvgPool(nn.Module):
    def forward(self, x): return x.mean((-2,-1))

In [14]:
def get_model2(act=nn.ReLU, nfs=(16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [ResBlock(256, 512, act=act, norm=norm), GlobalAvgPool()]
    layers += [nn.Linear(512, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [15]:
#|export
def _flops(x, h, w):
    if x.dim()<3: return x.numel()
    if x.dim()==4: return x.numel()*h*w

@fc.patch
def summary(self:Learner):
    res = '|Module|Input|Output|Num params|MFLOPS|\n|--|--|--|--|--|\n'
    totp,totf = 0,0
    def _f(hook, mod, inp, outp):
        nonlocal res,totp,totf
        nparms = sum(o.numel() for o in mod.parameters())
        totp += nparms
        *_,h,w = outp.shape
        flops = sum(_flops(o, h, w) for o in mod.parameters())/1e6
        totf += flops
        res += f'|{type(mod).__name__}|{tuple(inp[0].shape)}|{tuple(outp.shape)}|{nparms}|{flops:.1f}|\n'
    with Hooks(self.model, _f) as hooks: self.fit(1, lr=1, cbs=SingleBatchCB())
    print(f"Tot params: {totp}; MFLOPS: {totf:.1f}")
    if fc.IN_NOTEBOOK:
        from IPython.display import Markdown
        return Markdown(res)
    else: print(res)

In [16]:
TrainLearner(get_model2(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

Tot params: 4907588; MFLOPS: 33.0


|Module|Input|Output|Num params|MFLOPS|
|--|--|--|--|--|
|ResBlock|(1024, 1, 28, 28)|(1024, 16, 28, 28)|6928|5.3|
|ResBlock|(1024, 16, 28, 28)|(1024, 32, 14, 14)|14560|2.8|
|ResBlock|(1024, 32, 14, 14)|(1024, 64, 7, 7)|57792|2.8|
|ResBlock|(1024, 64, 7, 7)|(1024, 128, 4, 4)|230272|3.7|
|ResBlock|(1024, 128, 4, 4)|(1024, 256, 2, 2)|919296|3.7|
|ResBlock|(1024, 256, 2, 2)|(1024, 512, 2, 2)|3673600|14.7|
|GlobalAvgPool|(1024, 512, 2, 2)|(1024, 512)|0|0.0|
|Linear|(1024, 512)|(1024, 10)|5120|0.0|
|BatchNorm1d|(1024, 10)|(1024, 10)|20|0.0|


In [17]:
set_seed(42)
model = get_model2(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.020 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 518.812 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.020 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.812 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.822,0.715,0,train
0.862,0.548,0,eval
0.898,0.383,1,train
0.881,0.379,1,eval
0.920,0.269,2,train
0.903,0.301,2,eval
0.940,0.199,3,train
0.924,0.238,3,eval
0.961,0.142,4,train
0.929,0.226,4,eval


Storage check (checkpoint_write): artifacts 0.040 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.791 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.079 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.752 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.138 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.691 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.138 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.691 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.138 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.691 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.138 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.691 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.138 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.691 GB. Budget is not Drive quota.


In [18]:
def get_model3(act=nn.ReLU, nfs=(16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [GlobalAvgPool(), nn.Linear(256, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [19]:
TrainLearner(get_model3(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

Tot params: 1231428; MFLOPS: 18.3


|Module|Input|Output|Num params|MFLOPS|
|--|--|--|--|--|
|ResBlock|(1024, 1, 28, 28)|(1024, 16, 28, 28)|6928|5.3|
|ResBlock|(1024, 16, 28, 28)|(1024, 32, 14, 14)|14560|2.8|
|ResBlock|(1024, 32, 14, 14)|(1024, 64, 7, 7)|57792|2.8|
|ResBlock|(1024, 64, 7, 7)|(1024, 128, 4, 4)|230272|3.7|
|ResBlock|(1024, 128, 4, 4)|(1024, 256, 2, 2)|919296|3.7|
|GlobalAvgPool|(1024, 256, 2, 2)|(1024, 256)|0|0.0|
|Linear|(1024, 256)|(1024, 10)|2560|0.0|
|BatchNorm1d|(1024, 10)|(1024, 10)|20|0.0|


In [20]:
[o.shape for o in get_model3()[0].parameters()]

[torch.Size([16, 1, 5, 5]),
 torch.Size([16]),
 torch.Size([16]),
 torch.Size([16]),
 torch.Size([16, 16, 5, 5]),
 torch.Size([16]),
 torch.Size([16]),
 torch.Size([16]),
 torch.Size([16, 1, 1, 1]),
 torch.Size([16])]

In [21]:
set_seed(42)
model = get_model3(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.040 GB; configured budget 80.000 GB; additional estimate 0.093 GB; runtime free 518.727 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.040 GB; configured budget 80.000 GB; additional estimate 0.013 GB; runtime free 518.727 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.810,0.758,0,train
0.871,0.452,0,eval
0.895,0.400,1,train
0.902,0.347,1,eval
0.920,0.276,2,train
0.897,0.316,2,eval
0.939,0.207,3,train
0.925,0.242,3,eval
0.960,0.152,4,train
0.929,0.228,4,eval


Storage check (checkpoint_write): artifacts 0.045 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.721 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.055 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.712 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.070 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.696 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.070 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.695 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.070 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.695 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.070 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.695 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.070 GB; configured budget 80.000 GB; additional estimate 0.013 GB; runtime free 518.695 GB. Budget is not Drive quota.


In [22]:
def get_model4(act=nn.ReLU, nfs=(16,32,64,128,256), norm=nn.BatchNorm2d):
    layers = [conv(1, 16, ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [GlobalAvgPool(), nn.Linear(256, 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [23]:
[o.shape for o in get_model4()[0].parameters()]

[torch.Size([16, 1, 5, 5]),
 torch.Size([16]),
 torch.Size([16]),
 torch.Size([16])]

In [24]:
TrainLearner(get_model4(), dls, F.cross_entropy, lr=lr, cbs=[DeviceCB()]).summary()

Tot params: 1224948; MFLOPS: 13.3


|Module|Input|Output|Num params|MFLOPS|
|--|--|--|--|--|
|Sequential|(1024, 1, 28, 28)|(1024, 16, 28, 28)|448|0.3|
|ResBlock|(1024, 16, 28, 28)|(1024, 32, 14, 14)|14560|2.8|
|ResBlock|(1024, 32, 14, 14)|(1024, 64, 7, 7)|57792|2.8|
|ResBlock|(1024, 64, 7, 7)|(1024, 128, 4, 4)|230272|3.7|
|ResBlock|(1024, 128, 4, 4)|(1024, 256, 2, 2)|919296|3.7|
|GlobalAvgPool|(1024, 256, 2, 2)|(1024, 256)|0|0.0|
|Linear|(1024, 256)|(1024, 10)|2560|0.0|
|BatchNorm1d|(1024, 10)|(1024, 10)|20|0.0|


In [25]:
set_seed(42)
model = get_model4(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.045 GB; configured budget 80.000 GB; additional estimate 0.092 GB; runtime free 518.702 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.045 GB; configured budget 80.000 GB; additional estimate 0.013 GB; runtime free 518.702 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.804,0.782,0,train
0.870,0.477,0,eval
0.898,0.392,1,train
0.883,0.384,1,eval
0.920,0.277,2,train
0.886,0.355,2,eval
0.940,0.203,3,train
0.924,0.239,3,eval
0.961,0.149,4,train
0.928,0.232,4,eval


Storage check (checkpoint_write): artifacts 0.050 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.697 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.060 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.687 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.075 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.673 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.075 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.673 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.075 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.672 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.075 GB; configured budget 80.000 GB; additional estimate 0.023 GB; runtime free 518.672 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.075 GB; configured budget 80.000 GB; additional estimate 0.013 GB; runtime free 518.672 GB. Budget is not Drive quota.


## Data augmentation

After 20 epochs without augmentation:

```
{'accuracy': '0.999', 'loss': '0.012', 'epoch': 19, 'train': True}
{'accuracy': '0.924', 'loss': '0.284', 'epoch': 19, 'train': False}
```

With batchnorm, weight decay doesn't really regularize.

In [26]:
from torchvision import transforms

In [27]:
def tfm_batch(b, tfm_x=fc.noop, tfm_y = fc.noop): return tfm_x(b[0]),tfm_y(b[1])

tfms = nn.Sequential(transforms.RandomCrop(28, padding=4),
                     transforms.RandomHorizontalFlip())

augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[SingleBatchCB(), augcb])

In [28]:
learn.fit(1)

In [29]:
xb,yb = learn.batch
show_images(xb[:16], imsize=1.5)

In [30]:
#| export
@fc.patch
@fc.delegates(show_images)
def show_image_batch(self:Learner, max_n=9, cbs=None, **kwargs):
    self.fit(1, cbs=[SingleBatchCB()]+fc.L(cbs))
    show_images(self.batch[0][:max_n], **kwargs)

In [31]:
learn.show_image_batch(max_n=16, imsize=(1.5))

In [32]:
tfms = nn.Sequential(transforms.RandomCrop(28, padding=1),
                     transforms.RandomHorizontalFlip())
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

In [33]:
set_seed(42)
epochs = 20
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.050 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 518.679 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.050 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.679 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.764,0.878,0,train
0.824,0.603,0,eval
0.858,0.597,1,train
0.873,0.483,1,eval
0.876,0.477,2,train
0.855,0.506,2,eval
0.888,0.391,3,train
0.888,0.354,3,eval
0.893,0.341,4,train
0.873,0.409,4,eval


Storage check (checkpoint_write): artifacts 0.070 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.659 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.109 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.620 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.169 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.553 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.169 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.553 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.169 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.552 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.169 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.552 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.169 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.552 GB. Budget is not Drive quota.


A custom collation function could let you do per-item transformations.

In [34]:
mdl_path = Path('models')
mdl_path.mkdir(exist_ok=True)
torch.save(learn.model, mdl_path/'data_aug.pkl')

## Test time augmentation (TTA)

In [35]:
#| export
class CapturePreds(Callback):
    def before_fit(self, learn): self.all_inps,self.all_preds,self.all_targs = [],[],[]
    def after_batch(self, learn):
        self.all_inps. append(to_cpu(learn.batch[0]))
        self.all_preds.append(to_cpu(learn.preds))
        self.all_targs.append(to_cpu(learn.batch[1]))
    def after_fit(self, learn):
        self.all_preds,self.all_targs,self.all_inps = map(torch.cat, [self.all_preds,self.all_targs,self.all_inps])

In [36]:
#| export
@fc.patch
def capture_preds(self: Learner, cbs=None, inps=False):
    cp = CapturePreds()
    self.fit(1, train=False, cbs=[cp]+fc.L(cbs))
    res = cp.all_preds,cp.all_targs
    if inps: res = res+(cp.all_inps,)
    return res

In [37]:
ap1, at = learn.capture_preds()

accuracy,loss,epoch,train
0.942,0.177,0,eval


<Figure size 600x400 with 1 Axes>

In [38]:
ttacb = BatchTransformCB(partial(tfm_batch, tfm_x=TF.hflip), on_val=True)
ap2, at = learn.capture_preds(cbs=[ttacb])

accuracy,loss,epoch,train
0.943,0.177,0,eval


<Figure size 600x400 with 1 Axes>

In [39]:
ap1.shape,ap2.shape,at.shape

(torch.Size([10000, 10]), torch.Size([10000, 10]), torch.Size([10000]))

In [40]:
ap = torch.stack([ap1,ap2]).mean(0).argmax(1)

In [41]:
round((ap==at).float().mean().item(), 3)

0.944

## Random erase

In [42]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]

In [43]:
xm,xs = xbt.mean(),xbt.std()

In [44]:
xbt.min(), xbt.max()

(tensor(-0.80), tensor(2.06))

In [45]:
pct = 0.2

In [46]:
szx = int(pct*xbt.shape[-2])
szy = int(pct*xbt.shape[-1])
stx = int(random.random()*(1-pct)*xbt.shape[-2])
sty = int(random.random()*(1-pct)*xbt.shape[-1])
stx,sty,szx,szy

(14, 0, 5, 5)

In [47]:
init.normal_(xbt[:,:,stx:stx+szx,sty:sty+szy], mean=xm, std=xs);

In [48]:
show_images(xbt, imsize=1.5)

In [49]:
xbt.min(), xbt.max()

(tensor(-3.36), tensor(2.56))

In [50]:
#|export
def _rand_erase1(x, pct, xm, xs, mn, mx):
    szx = int(pct*x.shape[-2])
    szy = int(pct*x.shape[-1])
    stx = int(random.random()*(1-pct)*x.shape[-2])
    sty = int(random.random()*(1-pct)*x.shape[-1])
    init.normal_(x[:,:,stx:stx+szx,sty:sty+szy], mean=xm, std=xs)
    x.clamp_(mn, mx)

In [51]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
_rand_erase1(xbt, 0.2, xbt.mean(), xbt.std(), xbt.min(), xbt.max())
show_images(xbt, imsize=1.5)

In [52]:
xbt.mean(),xbt.std(),xbt.min(), xbt.max()

(tensor(0.09), tensor(1.04), tensor(-0.80), tensor(2.06))

In [53]:
#|export
def rand_erase(x, pct=0.2, max_num = 4):
    xm,xs,mn,mx = x.mean(),x.std(),x.min(),x.max()
    num = random.randint(0, max_num)
    for i in range(num): _rand_erase1(x, pct, xm, xs, mn, mx)
#     print(num)
    return x

In [54]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
rand_erase(xbt, 0.2, 4)
show_images(xbt, imsize=1.5)

In [55]:
#|export
class RandErase(nn.Module):
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct,self.max_num = pct,max_num
    def forward(self, x): return rand_erase(x, self.pct, self.max_num)

In [56]:
tfms = nn.Sequential(transforms.RandomCrop(28, padding=1),
                     transforms.RandomHorizontalFlip(),
                     RandErase())
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

In [57]:
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[DeviceCB(), SingleBatchCB(), augcb])
learn.fit(1)
xb,yb = learn.batch
show_images(xb[:16], imsize=1.5)

In [58]:
epochs = 50
lr = 2e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.090 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 518.491 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.090 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.491 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.756,0.888,0,train
0.832,0.578,0,eval
0.844,0.621,1,train
0.856,0.535,1,eval
0.863,0.532,2,train
0.872,0.437,2,eval
0.872,0.462,3,train
0.842,0.524,3,eval
0.879,0.405,4,train
0.882,0.358,4,eval


Storage check (checkpoint_write): artifacts 0.110 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.470 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.149 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.430 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.210 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.164 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.210 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.164 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.210 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.163 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.210 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.163 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.210 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.163 GB. Budget is not Drive quota.


## Random copy

In [59]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]

In [60]:
szx = int(pct*xbt.shape[-2])
szy = int(pct*xbt.shape[-1])
stx1 = int(random.random()*(1-pct)*xbt.shape[-2])
sty1 = int(random.random()*(1-pct)*xbt.shape[-1])
stx2 = int(random.random()*(1-pct)*xbt.shape[-2])
sty2 = int(random.random()*(1-pct)*xbt.shape[-1])
stx1,sty1,stx2,sty2,szx,szy

(1, 6, 21, 3, 5, 5)

In [61]:
xbt[:,:,stx1:stx1+szx,sty1:sty1+szy] = xbt[:,:,stx2:stx2+szx,sty2:sty2+szy]

In [62]:
show_images(xbt, imsize=1.5)

In [63]:
#|export
def _rand_copy1(x, pct):
    szx = int(pct*x.shape[-2])
    szy = int(pct*x.shape[-1])
    stx1 = int(random.random()*(1-pct)*x.shape[-2])
    sty1 = int(random.random()*(1-pct)*x.shape[-1])
    stx2 = int(random.random()*(1-pct)*x.shape[-2])
    sty2 = int(random.random()*(1-pct)*x.shape[-1])
    x[:,:,stx1:stx1+szx,sty1:sty1+szy] = x[:,:,stx2:stx2+szx,sty2:sty2+szy]

In [64]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
_rand_copy1(xbt, 0.2)
show_images(xbt, imsize=1.5)

In [65]:
#|export
def rand_copy(x, pct=0.2, max_num = 4):
    num = random.randint(0, max_num)
    for i in range(num): _rand_copy1(x, pct)
#     print(num)
    return x

In [66]:
xb,_ = next(iter(dls.train))
xbt = xb[:16]
rand_copy(xbt, 0.2, 4)
show_images(xbt, imsize=1.5)

In [67]:
#|export
class RandCopy(nn.Module):
    def __init__(self, pct=0.2, max_num=4):
        super().__init__()
        self.pct,self.max_num = pct,max_num
    def forward(self, x): return rand_copy(x, self.pct, self.max_num)

In [68]:
tfms = nn.Sequential(transforms.RandomCrop(28, padding=1),
                     transforms.RandomHorizontalFlip(),
                     RandCopy())
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

In [69]:
model = get_model()
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=[DeviceCB(), SingleBatchCB(), augcb])
learn.fit(1)
xb,yb = learn.batch
show_images(xb[:16], imsize=1.5)

In [70]:
set_seed(1)
epochs = 25
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.112 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 518.167 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.112 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.167 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.738,0.944,0,train
0.813,0.606,0,eval
0.832,0.667,1,train
0.837,0.551,1,eval
0.848,0.571,2,train
0.832,0.502,2,eval
0.859,0.492,3,train
0.832,0.545,3,eval
0.871,0.420,4,train
0.874,0.376,4,eval


Storage check (checkpoint_write): artifacts 0.131 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.142 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.171 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 518.102 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.231 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.985 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.231 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.985 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.231 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.985 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.231 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.985 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.231 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 517.985 GB. Budget is not Drive quota.


In [71]:
model2 = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn2 = TrainLearner(model2, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn2.fit(epochs)

Storage check (before_fit): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 518.008 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.132 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 518.008 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.738,0.940,0,train
0.812,0.644,0,eval
0.829,0.674,1,train
0.822,0.580,1,eval
0.847,0.578,2,train
0.834,0.528,2,eval
0.855,0.499,3,train
0.855,0.445,3,eval
0.870,0.418,4,train
0.851,0.452,4,eval


Storage check (checkpoint_write): artifacts 0.152 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.987 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.191 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.948 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.251 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.888 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.251 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.888 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.251 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.887 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.251 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.887 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.251 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 517.887 GB. Budget is not Drive quota.


In [72]:
mdl_path = Path('models')
torch.save(learn.model,  mdl_path/'randcopy1.pkl')
torch.save(learn2.model, mdl_path/'randcopy2.pkl')

In [73]:
cp1 = CapturePreds()
learn.fit(1, train=False, cbs=cp1)

accuracy,loss,epoch,train
0.944,0.166,0,eval


<Figure size 600x400 with 1 Axes>

In [74]:
cp2 = CapturePreds()
learn2.fit(1, train=False, cbs=cp2)

accuracy,loss,epoch,train
0.943,0.167,0,eval


<Figure size 600x400 with 1 Axes>

In [75]:
ap = torch.stack([cp1.all_preds,cp2.all_preds]).mean(0).argmax(1)

In [76]:
round((ap==cp1.all_targs).float().mean().item(), 3)

0.945

## Dropout

In [77]:
p = 0.1
dist = distributions.binomial.Binomial(probs=1-p)
dist.sample((10,))

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [78]:
class Dropout(nn.Module):
    def __init__(self, p=0.1):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training: return x
        dist = distributions.binomial.Binomial(tensor(1.0).to(x.device), probs=1-self.p)
        return x * dist.sample(x.size()) * 1/(1-self.p)

In [79]:
def get_dropmodel(act=nn.ReLU, nfs=(16,32,64,128,256,512), norm=nn.BatchNorm2d, drop=0.0):
    layers = [ResBlock(1, 16, ks=5, stride=1, act=act, norm=norm), nn.Dropout2d(drop)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [nn.Flatten(), Dropout(drop), nn.Linear(nfs[-1], 10, bias=False), nn.BatchNorm1d(10)]
    return nn.Sequential(*layers)

In [80]:
set_seed(42)
epochs=5
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)

In [81]:
learn.fit(epochs)

Storage check (before_fit): artifacts 0.192 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 517.804 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.192 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 517.804 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.809,0.745,0,train
0.865,0.462,0,eval
0.893,0.392,1,train
0.897,0.321,1,eval
0.917,0.276,2,train
0.913,0.278,2,eval
0.937,0.209,3,train
0.927,0.237,3,eval
0.956,0.156,4,train
0.930,0.225,4,eval


Storage check (checkpoint_write): artifacts 0.212 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.784 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.251 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.745 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.311 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.685 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.311 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.685 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.311 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.685 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.311 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.685 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.311 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 517.685 GB. Budget is not Drive quota.


In [82]:
class TTD_CB(Callback):
    def before_epoch(self, learn):
        learn.model.apply(lambda m: m.train() if isinstance(m, (nn.Dropout,nn.Dropout2d)) else None)

## Augment 2

In [83]:
@inplace
def transformi(b): b[xl] = [(TF.to_tensor(o)*2-1) for o in b[xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

In [84]:
set_seed(42)
epochs = 20
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched), augcb]
model = get_model(act_gr, norm=nn.BatchNorm2d).apply(iw)
learn = TrainLearner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=optim.AdamW)
learn.fit(epochs)

Storage check (before_fit): artifacts 0.212 GB; configured budget 80.000 GB; additional estimate 0.269 GB; runtime free 517.724 GB. Budget is not Drive quota.
Storage check (checkpoint_write): artifacts 0.212 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 517.724 GB. Budget is not Drive quota.


accuracy,loss,epoch,train
0.742,0.932,0,train
0.797,0.627,0,eval
0.832,0.662,1,train
0.780,0.721,1,eval
0.856,0.534,2,train
0.850,0.492,2,eval
0.849,0.493,3,train
0.838,0.469,3,eval
0.869,0.404,4,train
0.842,0.442,4,eval


Storage check (checkpoint_write): artifacts 0.232 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.703 GB. Budget is not Drive quota.


<Figure size 600x400 with 1 Axes>

Storage check (checkpoint_write): artifacts 0.271 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.664 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 0.331 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.228 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.331 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.228 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.331 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.227 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.331 GB; configured budget 80.000 GB; additional estimate 0.067 GB; runtime free 517.227 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 0.331 GB; configured budget 80.000 GB; additional estimate 0.028 GB; runtime free 517.227 GB. Budget is not Drive quota.


In [85]:
torch.save(learn.model, 'models/data_aug2.pkl')

### Export -

In [86]:
# The bundled miniai package is ready to import; no export is required.

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [87]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [88]:
# NativeNext independent numerical probe.
_next.probe(globals())


__HAILONG_NATIVE_NEXT_FULL_EVIDENCE__{"contract": {"passed": true, "target": "04_fastai_part2/14_augment.ipynb", "ordinary_fits": 10, "ordinary_epochs": 165, "ordinary_train_batches": 9735, "ordinary_valid_batches": 825, "training_single_batch_demos": 7, "full_prediction_captures": 4}, "numerical_artifact_subprobe_only": true, "numerical_models": [{"cell_index": 14, "original_epochs": 5, "architecture": {"kind": "original_residual_augmentation_comparison", "residual_blocks": 6, "classifier_features": 512, "dropout_probability": 0.0}, "probe_rows": 64, "cross_entropy": 0.26873213052749634, "accuracy": 0.921875, "model_checkpoint_snapshot_equal": true, "exact_saved_inference_equal": true, "activation_statistics_retained": true, "optimizer_strict_roundtrip": true, "preserved_gradients_restored": true, "rng_validation": {"python_numpy_torch_cpu_generator_restore": true, "cuda_state_devices": 1, "cuda_validation": "saved ByteTensor structure only; no GPU generator replay claim"}}, {"cell_in

{'contract': {'passed': True,
  'target': '04_fastai_part2/14_augment.ipynb',
  'ordinary_fits': 10,
  'ordinary_epochs': 165,
  'ordinary_train_batches': 9735,
  'ordinary_valid_batches': 825,
  'training_single_batch_demos': 7,
  'full_prediction_captures': 4},
 'numerical_artifact_subprobe_only': True,
 'numerical_models': [{'cell_index': 14,
   'original_epochs': 5,
   'architecture': {'kind': 'original_residual_augmentation_comparison',
    'residual_blocks': 6,
    'classifier_features': 512,
    'dropout_probability': 0.0},
   'probe_rows': 64,
   'cross_entropy': 0.26873213052749634,
   'accuracy': 0.921875,
   'model_checkpoint_snapshot_equal': True,
   'exact_saved_inference_equal': True,
   'activation_statistics_retained': True,
   'optimizer_strict_roundtrip': True,
   'preserved_gradients_restored': True,
   'rng_validation': {'python_numpy_torch_cpu_generator_restore': True,
    'cuda_state_devices': 1,
    'cuda_validation': 'saved ByteTensor structure only; no GPU gene